# Week 12 Mini Graded Project — E-Commerce Order, Refund & Exception Handling with CrewAI

This notebook is a **clean, runnable version** of the Week 12 graded mini project using **CrewAI** with a **Together provider**.

## What is improved in this version?
- Uses Together API correctly with CrewAI
- Avoids the `ValidationError` caused by passing `ChatTogether(...)` directly into `Agent(llm=...)`
- Uses **separate code blocks for each agent**, so it is easier to understand and run
- Uses a simpler dependency setup
- Includes representative test cases
- Includes architecture explanation for submission

## Use Case Chosen
**E-Commerce — Order, Refund & Exception Handling**

## Objective
Build a realistic multi-agent system that can:
- identify customer order issues
- interpret return/refund policies
- recommend a resolution
- decide whether the case should be escalated


## Step 1 — Install Required Libraries

We install only the minimum required libraries to reduce dependency conflicts.

### Important note
If you previously installed many conflicting versions in the same Colab runtime, first do:
- **Runtime → Restart session**
and then run this notebook from the top.


In [ ]:
!pip install -q crewai litellm python-dotenv

## Step 2 — Load Environment Variables

This notebook uses a Together API key from your `.env` file.

### Example `.env`
```env
LLM_PROVIDER=together
TOGETHER_API_KEY=your_key_here
LLM_MODEL_DEFAULT=Qwen/Qwen2.5-7B-Instruct-Turbo
```


In [ ]:
import os
import json
import warnings
from pathlib import Path
from dotenv import load_dotenv

warnings.filterwarnings('ignore')

def _load_env():
    cwd = Path.cwd()
    for parent in [cwd] + list(cwd.parents):
        env_path = parent / '.env'
        if env_path.exists():
            load_dotenv(env_path)
            return env_path
    load_dotenv()
    return None

env_path = _load_env()
print(f'.env loaded from: {env_path}')

PROVIDER = os.getenv('LLM_PROVIDER', 'together').lower()
print('Provider:', PROVIDER)

if PROVIDER != 'together':
    raise ValueError('This notebook is configured for Together provider only.')

together_api_key = os.getenv('TOGETHER_API_KEY')
if not together_api_key:
    raise ValueError('TOGETHER_API_KEY is missing from your .env file')

model_name = os.getenv('LLM_MODEL_DEFAULT', 'Qwen/Qwen2.5-7B-Instruct-Turbo')

# CrewAI / LiteLLM compatibility
os.environ['TOGETHERAI_API_KEY'] = together_api_key

# IMPORTANT FIX: pass a string LLM reference to CrewAI agents
crewai_llm = f'together_ai/{model_name}'

print('Model name:', model_name)
print('CrewAI LLM reference:', crewai_llm)

## Step 3 — Import CrewAI Classes

We import the core classes used to define the workflow.


In [ ]:
from crewai import Agent, Task, Crew

## Step 4 — Define E-Commerce Policy Rules

Since RAG is not required, we use static business rules as the knowledge source.


In [ ]:
ECOMMERCE_POLICY = '''
E-Commerce Policy Rules:

1. Standard return window:
   - Items can be returned within 7 days of delivery.

2. Damaged item policy:
   - If the item arrives damaged, the customer is eligible for replacement or refund.

3. Wrong item policy:
   - If the wrong item is delivered, the customer is eligible for replacement or refund.

4. Delayed shipment policy:
   - If an order is not delivered and is delayed by more than 5 days beyond expected delivery,
     the customer may cancel for a refund.
   - If still in transit and delay is small, advise waiting.

5. Non-returnable items:
   - Grocery items
   - Personal care products
   - Custom-made products

6. Opened electronics:
   - If damaged or wrong item, replacement is allowed.
   - Refund may require manual review.

7. High-value review threshold:
   - Orders above $500 must be manually reviewed before final refund approval.

8. Delivered but not received:
   - Must be escalated for manual investigation.

9. Repeat refund behavior:
   - More than 2 refund requests in the last 30 days should be escalated.

10. Missing information:
   - If the complaint lacks enough detail, ask for clarification before resolution.

11. Resolution types allowed:
   - Refund
   - Replacement
   - Wait / monitor shipment
   - Reject request
   - Ask for more information
   - Escalate for manual review
'''

print(ECOMMERCE_POLICY[:500] + '\n...')

## Step 5 — Create Mock Order Database

We simulate e-commerce order records using static mock data.


In [ ]:
mock_orders = {
    '1001': {
        'customer_name': 'Aarav',
        'item': 'Wireless Earbuds',
        'category': 'electronics',
        'price': 80,
        'status': 'in_transit',
        'expected_delivery_days_late': 6,
        'delivered_days_ago': None,
        'opened': False,
        'prior_refund_requests_30d': 0
    },
    '1002': {
        'customer_name': 'Meera',
        'item': 'Blender',
        'category': 'home_appliance',
        'price': 120,
        'status': 'delivered',
        'expected_delivery_days_late': 0,
        'delivered_days_ago': 0,
        'opened': True,
        'prior_refund_requests_30d': 0
    },
    '1003': {
        'customer_name': 'Rahul',
        'item': 'Running Shoes',
        'category': 'fashion',
        'price': 65,
        'status': 'delivered',
        'expected_delivery_days_late': 0,
        'delivered_days_ago': 0,
        'opened': False,
        'prior_refund_requests_30d': 0
    },
    '1004': {
        'customer_name': 'Sana',
        'item': 'Office Chair',
        'category': 'furniture',
        'price': 150,
        'status': 'delivered',
        'expected_delivery_days_late': 0,
        'delivered_days_ago': 10,
        'opened': False,
        'prior_refund_requests_30d': 0
    },
    '1005': {
        'customer_name': 'Kiran',
        'item': 'Skin Care Cream',
        'category': 'personal_care',
        'price': 25,
        'status': 'delivered',
        'expected_delivery_days_late': 0,
        'delivered_days_ago': 2,
        'opened': True,
        'prior_refund_requests_30d': 0
    },
    '1006': {
        'customer_name': 'Nisha',
        'item': 'Laptop',
        'category': 'electronics',
        'price': 1200,
        'status': 'delivered',
        'expected_delivery_days_late': 0,
        'delivered_days_ago': 1,
        'opened': True,
        'prior_refund_requests_30d': 1
    },
    '1007': {
        'customer_name': 'Vikram',
        'item': 'Smartphone',
        'category': 'electronics',
        'price': 700,
        'status': 'delivered',
        'expected_delivery_days_late': 0,
        'delivered_days_ago': 1,
        'opened': False,
        'prior_refund_requests_30d': 0
    }
}

print('Mock orders loaded:', len(mock_orders))

## Step 6 — Helper Function

This helper returns order details in structured JSON text format.


In [ ]:
def get_order_context(order_id):
    if order_id in mock_orders:
        return json.dumps(mock_orders[order_id], indent=2)
    return 'Order ID not found in mock database.'

# Agent Creation Section

Below, each agent is defined in a **separate code block** so it is easier to run and debug.


## Step 7 — Agent 1: Order Issue Identification Agent

This agent reads the complaint and extracts structured case facts.


In [ ]:
issue_agent = Agent(
    role='Order Issue Identification Agent',
    goal='Identify the exact customer issue, extract key facts, and structure the case for downstream agents.',
    backstory=(
        'You are an e-commerce support triage specialist. '
        'You read customer complaints carefully, identify issue types such as delay, damage, '
        'wrong item, refund request, or ambiguity, and produce structured case summaries.'
    ),
    llm=crewai_llm,
    verbose=True
)

print('Issue agent created successfully')

## Step 8 — Agent 2: Policy Interpretation Agent

This agent applies static business policy rules to the extracted case facts.


In [ ]:
policy_agent = Agent(
    role='Policy Interpretation Agent',
    goal='Apply return, refund, delay, and exception policies consistently using the structured issue summary and order details.',
    backstory=(
        'You are an expert in e-commerce business rules and policy enforcement. '
        'You interpret fixed company policies fairly and consistently.'
    ),
    llm=crewai_llm,
    verbose=True
)

print('Policy agent created successfully')

## Step 9 — Agent 3: Resolution Recommendation Agent

This agent converts policy results into an actionable support response.


In [ ]:
resolution_agent = Agent(
    role='Resolution Recommendation Agent',
    goal='Recommend the most appropriate customer resolution using the issue summary and policy decision.',
    backstory=(
        'You are a customer resolution expert who converts policy outcomes into practical next steps, '
        'including refund, replacement, wait guidance, rejection, or request for more information.'
    ),
    llm=crewai_llm,
    verbose=True
)

print('Resolution agent created successfully')

## Step 10 — Agent 4: Escalation Agent

This agent decides whether manual review is required.


In [ ]:
escalation_agent = Agent(
    role='Escalation Agent',
    goal='Decide whether the case requires manual review based on risk, urgency, cost, uncertainty, or policy exceptions.',
    backstory=(
        'You are responsible for safeguarding customer support decisions by escalating risky, uncertain, '
        'high-value, suspicious, or policy-sensitive cases for human review.'
    ),
    llm=crewai_llm,
    verbose=True
)

print('Escalation agent created successfully')

## Step 11 — Define Tasks and Handoffs

Workflow:
Customer Query + Order Context → Issue Identification → Policy Interpretation → Resolution Recommendation → Escalation Decision


In [ ]:
issue_task = Task(
    description='''
You are given:
1. Customer query: {customer_query}
2. Order ID: {order_id}
3. Order details: {order_context}

Your job:
- Identify the main issue type:
  delayed_delivery / damaged_item / wrong_item / return_request / refund_request / not_received / unclear
- Extract important facts:
  - delivery status
  - days late or delivered days ago
  - item category
  - item value
  - whether the customer wants refund/replacement/help
  - whether key information is missing
- Estimate urgency as: low / medium / high
- Return a structured summary.

Output format:
Issue Type:
Key Facts:
Customer Intent:
Urgency:
Missing Information:
''',
    expected_output='A structured issue identification summary.',
    agent=issue_agent
)

policy_task = Task(
    description='''
You are given:
1. Customer query: {customer_query}
2. Order details: {order_context}
3. Company policy:
{policy_text}

Use the issue-identification output from the previous agent.

Your job:
- Apply the policy to the case
- Determine:
  - eligible / not eligible / partially eligible
  - allowed resolution types
  - whether any exception applies
  - whether policy interpretation is uncertain
- Justify the result based on the policy rules

Output format:
Policy Decision:
Eligibility:
Applicable Rules:
Allowed Resolution Types:
Exception Notes:
Policy Uncertainty:
''',
    expected_output='A policy interpretation summary with eligibility and applicable rules.',
    agent=policy_agent
)

resolution_task = Task(
    description='''
You are given:
1. Customer query: {customer_query}
2. Order details: {order_context}
3. Policy text: {policy_text}

Use the outputs from:
- Issue Identification Agent
- Policy Interpretation Agent

Your job:
Recommend the best next step for the customer.

Possible actions:
- approve_refund
- approve_replacement
- advise_wait
- deny_request
- ask_for_more_info
- conditional_refund
- conditional_replacement

Output format:
Recommended Resolution:
Reasoning:
Customer Message Draft:
Operational Next Step:
''',
    expected_output='A clear resolution recommendation and customer-facing message draft.',
    agent=resolution_agent
)

escalation_task = Task(
    description='''
You are given:
1. Customer query: {customer_query}
2. Order details: {order_context}
3. Policy text: {policy_text}

Use all previous agent outputs.

Escalate if any of the following apply:
- order value > 500
- delivered but customer claims not received
- policy uncertainty exists
- repeat refund requests > 2 in last 30 days
- opened electronics refund requires manual review
- conflicting facts or missing important evidence
- explicit high-risk or exceptional complaint

Your job:
- Decide if escalation is required: yes / no
- Explain why
- Provide final case decision

Output format:
Escalation Required:
Escalation Reason:
Final Decision:
Final Action Owner:
''',
    expected_output='A final escalation decision with justification.',
    agent=escalation_agent
)

print('Tasks created successfully')

## Step 12 — Build the Crew

Now we combine all agents and tasks into one sequential pipeline.


In [ ]:
support_crew = Crew(
    agents=[issue_agent, policy_agent, resolution_agent, escalation_agent],
    tasks=[issue_task, policy_task, resolution_task, escalation_task],
    verbose=2
)

print('Crew pipeline ready')

## Step 13 — Runner Function

This function runs the workflow for one user query.


In [ ]:
def run_case(order_id, customer_query):
    order_context = get_order_context(order_id)

    inputs = {
        'order_id': order_id,
        'customer_query': customer_query,
        'order_context': order_context,
        'policy_text': ECOMMERCE_POLICY
    }

    result = support_crew.kickoff(inputs=inputs)

    print('\n' + '='*90)
    print(f'ORDER ID: {order_id}')
    print(f'QUERY: {customer_query}')
    print('-'*90)
    print('FINAL OUTPUT:')
    print(result)
    print('='*90 + '\n')

    return result

## Step 14 — Representative Test Cases

These sample inputs include both normal and edge cases.


In [ ]:
test_cases = [
    ('1001', 'My order #1001 was supposed to arrive 6 days ago and still has not been delivered. I want to cancel and get a refund.'),
    ('1002', 'I received order #1002 today, but the blender jar is cracked and unusable. I want a replacement.'),
    ('1003', 'Order #1003 delivered today, but I ordered black shoes and got white shoes. Please fix this.'),
    ('1004', 'I got order #1004 ten days ago and now I want to return it because I changed my mind.'),
    ('1005', 'I want to return the skin-care product from order #1005 because I do not like it.'),
    ('1006', 'My order #1006 laptop arrived damaged and the screen is broken. I need a refund immediately.'),
    ('1007', 'Tracking says my order #1007 was delivered, but I never got it.'),
    ('9999', 'My order is wrong and I want my money back.')
]

print('Total test cases:', len(test_cases))

## Step 15 — Run One Sample Test

Run this first to confirm the pipeline is working.


In [ ]:
run_case(*test_cases[0])

## Step 16 — Run All Test Cases

This demonstrates the end-to-end system behavior across multiple scenarios.


In [ ]:
all_results = []
for case in test_cases:
    result = run_case(*case)
    all_results.append(result)

# Architecture Explanation

## 1. Overview
This project implements a goal-oriented multi-agent workflow for e-commerce customer support using CrewAI. The system handles customer issues related to delayed deliveries, damaged items, wrong items, refund eligibility, and policy exceptions. The workflow is designed to replicate real operational reasoning by dividing the process across specialized agents that collaborate through structured handoffs.

The main objective is to provide fast, consistent, policy-compliant resolutions while ensuring risky or uncertain cases are escalated for human review.

## 2. Agent Roles
### Agent 1: Order Issue Identification Agent
This agent reads the customer query and order context, then identifies the core issue category such as delayed delivery, damaged item, wrong item, return request, or unclear complaint. It also extracts operational facts such as delivery status, item category, days delayed, item value, and customer intent. Its output is a structured summary used by downstream agents.

### Agent 2: Policy Interpretation Agent
This agent applies a predefined static e-commerce policy to the structured case summary and order data. It determines whether the customer is eligible for refund, replacement, return, or rejection. It also identifies exceptions such as non-returnable items, out-of-window returns, high-value purchases, or uncertainty in policy application.

### Agent 3: Resolution Recommendation Agent
This agent converts the issue summary and policy outcome into an actionable recommendation. It decides the most appropriate next step such as approve refund, approve replacement, advise waiting, deny request, or request more information. It also drafts a customer-facing response.

### Agent 4: Escalation Agent
This agent performs risk control. It evaluates whether the case should be manually escalated based on predefined rules such as high-value items, delivered-but-missing complaints, policy uncertainty, repeated refunds, or opened electronics requiring special approval.

## 3. Task Flow / Handoffs
The workflow follows a sequential collaboration pattern:
1. Customer complaint and order data enter the system
2. Issue Identification Agent extracts and structures facts
3. Policy Interpretation Agent applies rules using those extracted facts
4. Resolution Recommendation Agent proposes the operational resolution
5. Escalation Agent makes the final decision on human review

This ensures each agent depends on previous outputs rather than acting independently. The handoff structure improves clarity, traceability, and modularity.

## 4. Escalation Logic
Escalation is a deliberate and explicit part of the workflow. A case is escalated if:
- the order value is above $500
- tracking says delivered but the customer claims non-receipt
- the policy result is uncertain
- there are more than 2 prior refund requests in 30 days
- the case involves opened electronics needing refund approval
- facts are conflicting or important details are missing

This design reduces false approvals and reflects realistic support governance.

## 5. Sample Inputs and Outputs
Representative queries include delayed shipment, damaged items, wrong item delivery, non-returnable product returns, out-of-window returns, high-value damaged electronics, and delivered-but-missing cases. Low-risk cases typically result in direct refund/replacement/wait decisions, while exceptional or expensive cases trigger manual escalation.

Examples:
- delayed shipment more than 5 days → refund possible
- damaged blender → replacement approved
- skincare product return → denied as non-returnable
- damaged laptop worth $1200 → escalate for manual review
- delivered but not received → escalate for investigation

## 6. Design Rationale
The design follows the core principles of goal-oriented multi-agent systems:
- specialized agent roles instead of one general agent
- structured handoffs between agents
- static policy-based reasoning rather than RAG
- clear escalation thresholds
- realistic operational behavior

This makes the solution easy to explain, test, and extend. Additional policies, fraud checks, or loyalty-tier logic can be added later without redesigning the whole system.
